# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os
import getpass
import duckdb
import pandas as pd
import numpy as np

In [2]:
con = duckdb.connect()
hf_token = getpass.getpass("Enter your Hugging Face READ token: ")

Enter your Hugging Face READ token: ··········


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Rule: prioritize high-volume search opportunities

I will prioritize content items that already receive meaningful search impressions but have relatively weak click-through performance.

The rule is based on two signals:

1. **Search volume** — February GSC impressions. This is linked to the FlyRank quick-win idea: higher volume creates more potential value from an improvement.
2. **CTR vs position** — February GSC clicks and impressions are used to calculate CTR, while average position provides context for whether weak clicks are surprising for the item's search visibility.

**Reason code:** `CTR_OPPORTUNITY`

**Action label:** `REVIEW_CTR`

The score is intended to rank content for human review, not to claim that the rule proves future performance.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [3]:
# Configure DuckDB for temporary HTTP failures
con.execute("SET http_retries = 3")
con.execute("SET http_timeout = 30")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"
FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"

print("February 2026 data path ready")

February 2026 data path ready


In [5]:
# Register the Hugging Face token with DuckDB
con.execute("SET VARIABLE hf_token = ?", [hf_token])

con.execute("""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN getvariable('hf_token')
    )
""")

print("Hugging Face authentication configured")

Hugging Face authentication configured


In [6]:
# Build February client-content signal frame
signal_frame = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS gsc_clicks,
        SUM(gsc_impressions) AS gsc_impressions,
        AVG(gsc_avg_position) AS avg_position
    FROM {FEB}
    GROUP BY client_hash_id, content_hash_id
""").df()

# Calculate CTR only where impressions are positive
signal_frame["ctr"] = np.where(
    signal_frame["gsc_impressions"] > 0,
    signal_frame["gsc_clicks"] / signal_frame["gsc_impressions"],
    np.nan
)

print("Signal frame created:", signal_frame.shape)

signal_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Signal frame created: (321546, 6)


,client_hash_id,content_hash_id,gsc_clicks,gsc_impressions,avg_position,ctr
0,client_3ffa76342f366962,content_fb84747a57b8b665,0.0,0.0,NaN,NaN
1,client_3ffa76342f366962,content_feccf822ac21326e,0.0,0.0,NaN,NaN
2,client_3ffa76342f366962,content_17cf93c10413ebe9,0.0,0.0,NaN,NaN
3,client_3ffa76342f366962,content_a9905735266f8697,0.0,0.0,NaN,NaN
4,client_3ffa76342f366962,content_31c34765e7bba2f0,0.0,0.0,NaN,NaN


In [7]:
# Build the baseline action score.
# Higher score = higher priority for human review.

signal_frame["volume_score"] = np.log1p(signal_frame["gsc_impressions"])

signal_frame["ctr_gap"] = (
    signal_frame["ctr"].median() - signal_frame["ctr"]
)

signal_frame["action_score"] = (
    signal_frame["volume_score"] * signal_frame["ctr_gap"]
)

# Only keep finite scores for ranking.
ranked_queue = signal_frame[
    np.isfinite(signal_frame["action_score"])
].copy()

# Highest-priority opportunities first.
ranked_queue = ranked_queue.sort_values(
    "action_score",
    ascending=False
).reset_index(drop=True)

# Add the reason code and action label.
ranked_queue["reason_code"] = "CTR_OPPORTUNITY"
ranked_queue["action_label"] = "REVIEW_CTR"

# Save the ranked queue.
output_path = "work/outputs/baseline_action_score.csv"

os.makedirs("work/outputs", exist_ok=True)

ranked_queue.to_csv(output_path, index=False)

print("Ranked queue created:", ranked_queue.shape)
print("Saved to:", output_path)

ranked_queue.head(10)

Ranked queue created: (153559, 11)
Saved to: work/outputs/baseline_action_score.csv


,client_hash_id,content_hash_id,gsc_clicks,gsc_impressions,avg_position,ctr,volume_score,ctr_gap,action_score,reason_code,action_label
0,client_a80fca3f171ed1de,content_6d39e678ecda2135,0.0,11.0,2.272727,0.0,2.484907,0.0,0.0,CTR_OPPORTUNITY,REVIEW_CTR
1,client_3ffa76342f366962,content_c2eed5ce37894647,0.0,2.0,23.000000,0.0,1.098612,0.0,0.0,CTR_OPPORTUNITY,REVIEW_CTR
2,client_3ffa76342f366962,content_36bee0a093d0711d,0.0,3.0,5.500000,0.0,1.386294,0.0,0.0,CTR_OPPORTUNITY,REVIEW_CTR
3,client_3ffa76342f366962,content_04bed7e232ca42cd,0.0,1.0,9.000000,0.0,0.693147,0.0,0.0,CTR_OPPORTUNITY,REVIEW_CTR
4,client_3ffa76342f366962,content_0239c0a89e9454c5,0.0,1.0,47.000000,0.0,0.693147,0.0,0.0,CTR_OPPORTUNITY,REVIEW_CTR
5,client_3ffa76342f366962,content_4e2d90da35481342,0.0,3.0,9.000000,0.0,1.386294,0.0,0.0,CTR_OPPORTUNITY,REVIEW_CTR
6,client_3ffa76342f366962,content_cfbf60a6d1558926,0.0,1.0,12.000000,0.0,0.693147,0.0,0.0,CTR_OPPORTUNITY,REVIEW_CTR
7,client_3ffa76342f366962,content_22c76a7e96510f57,0.0,1.0,9.000000,0.0,0.693147,0.0,0.0,CTR_OPPORTUNITY,REVIEW_CTR
8,client_3ffa76342f366962,content_5736779b45111bab,0.0,1.0,9.000000,0.0,0.693147,0.0,0.0,CTR_OPPORTUNITY,REVIEW_CTR
9,client_3ffa76342f366962,content_8afd5f61282636ce,0.0,2.0,1.500000,0.0,1.098612,0.0,0.0,CTR_OPPORTUNITY,REVIEW_CTR


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [8]:
# Review the top 20 ranked opportunities.
top20 = ranked_queue.head(20).copy()

print("Top-20 review:")
display(
    top20[
        [
            "client_hash_id",
            "content_hash_id",
            "gsc_clicks",
            "gsc_impressions",
            "avg_position",
            "ctr",
            "volume_score",
            "ctr_gap",
            "action_score",
            "reason_code",
            "action_label",
        ]
    ]
)

Top-20 review:


,client_hash_id,content_hash_id,gsc_clicks,gsc_impressions,avg_position,ctr,volume_score,ctr_gap,action_score,reason_code,action_label
0,client_a80fca3f171ed1de,content_6d39e678ecda2135,0.0,11.0,2.272727,0.0,2.484907,0.0,0.0,CTR_OPPORTUNITY,REVIEW_CTR
1,client_3ffa76342f366962,content_c2eed5ce37894647,0.0,2.0,23.000000,0.0,1.098612,0.0,0.0,CTR_OPPORTUNITY,REVIEW_CTR
2,client_3ffa76342f366962,content_36bee0a093d0711d,0.0,3.0,5.500000,0.0,1.386294,0.0,0.0,CTR_OPPORTUNITY,REVIEW_CTR
3,client_3ffa76342f366962,content_04bed7e232ca42cd,0.0,1.0,9.000000,0.0,0.693147,0.0,0.0,CTR_OPPORTUNITY,REVIEW_CTR
4,client_3ffa76342f366962,content_0239c0a89e9454c5,0.0,1.0,47.000000,0.0,0.693147,0.0,0.0,CTR_OPPORTUNITY,REVIEW_CTR
5,client_3ffa76342f366962,content_4e2d90da35481342,0.0,3.0,9.000000,0.0,1.386294,0.0,0.0,CTR_OPPORTUNITY,REVIEW_CTR
6,client_3ffa76342f366962,content_cfbf60a6d1558926,0.0,1.0,12.000000,0.0,0.693147,0.0,0.0,CTR_OPPORTUNITY,REVIEW_CTR
7,client_3ffa76342f366962,content_22c76a7e96510f57,0.0,1.0,9.000000,0.0,0.693147,0.0,0.0,CTR_OPPORTUNITY,REVIEW_CTR
8,client_3ffa76342f366962,content_5736779b45111bab,0.0,1.0,9.000000,0.0,0.693147,0.0,0.0,CTR_OPPORTUNITY,REVIEW_CTR
9,client_3ffa76342f366962,content_8afd5f61282636ce,0.0,2.0,1.500000,0.0,1.098612,0.0,0.0,CTR_OPPORTUNITY,REVIEW_CTR


In [9]:
# Create the required Top-20 human review table.

top20_review = top20.copy()

top20_review["confidence_note"] = np.where(
    top20_review["gsc_impressions"] >= 10,
    "Medium confidence: enough search impressions for a basic volume signal.",
    "Low confidence: very few impressions, so the ranking is unstable."
)

top20_review["what_would_make_it_wrong"] = np.where(
    top20_review["gsc_impressions"] < 10,
    "Low search volume may make this look like an opportunity by chance.",
    "CTR may be unavailable or too weak to confirm a real opportunity."
)

display(
    top20_review[
        [
            "client_hash_id",
            "content_hash_id",
            "action_label",
            "reason_code",
            "action_score",
            "confidence_note",
            "what_would_make_it_wrong",
        ]
    ]
)

,client_hash_id,content_hash_id,action_label,reason_code,action_score,confidence_note,what_would_make_it_wrong
0,client_a80fca3f171ed1de,content_6d39e678ecda2135,REVIEW_CTR,CTR_OPPORTUNITY,0.0,Medium confidence: enough search impressions f...,CTR may be unavailable or too weak to confirm ...
1,client_3ffa76342f366962,content_c2eed5ce37894647,REVIEW_CTR,CTR_OPPORTUNITY,0.0,"Low confidence: very few impressions, so the r...",Low search volume may make this look like an o...
2,client_3ffa76342f366962,content_36bee0a093d0711d,REVIEW_CTR,CTR_OPPORTUNITY,0.0,"Low confidence: very few impressions, so the r...",Low search volume may make this look like an o...
3,client_3ffa76342f366962,content_04bed7e232ca42cd,REVIEW_CTR,CTR_OPPORTUNITY,0.0,"Low confidence: very few impressions, so the r...",Low search volume may make this look like an o...
4,client_3ffa76342f366962,content_0239c0a89e9454c5,REVIEW_CTR,CTR_OPPORTUNITY,0.0,"Low confidence: very few impressions, so the r...",Low search volume may make this look like an o...
5,client_3ffa76342f366962,content_4e2d90da35481342,REVIEW_CTR,CTR_OPPORTUNITY,0.0,"Low confidence: very few impressions, so the r...",Low search volume may make this look like an o...
6,client_3ffa76342f366962,content_cfbf60a6d1558926,REVIEW_CTR,CTR_OPPORTUNITY,0.0,"Low confidence: very few impressions, so the r...",Low search volume may make this look like an o...
7,client_3ffa76342f366962,content_22c76a7e96510f57,REVIEW_CTR,CTR_OPPORTUNITY,0.0,"Low confidence: very few impressions, so the r...",Low search volume may make this look like an o...
8,client_3ffa76342f366962,content_5736779b45111bab,REVIEW_CTR,CTR_OPPORTUNITY,0.0,"Low confidence: very few impressions, so the r...",Low search volume may make this look like an o...
9,client_3ffa76342f366962,content_8afd5f61282636ce,REVIEW_CTR,CTR_OPPORTUNITY,0.0,"Low confidence: very few impressions, so the r...",Low search volume may make this look like an o...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [11]:
# Weak picks + explicit leakage check

weak_picks = top20_review[
    (top20_review["gsc_impressions"] < 10) |
    (top20_review["action_score"] == 0)
].copy()

print("Weak picks in Top-20:", len(weak_picks))

display(
    weak_picks[
        [
            "client_hash_id",
            "content_hash_id",
            "gsc_impressions",
            "ctr",
            "action_score",
            "reason_code",
            "confidence_note",
            "what_would_make_it_wrong",
        ]
    ]
)

# Explicitly check for columns that would contain future March/outcome data.
future_columns = [
    col for col in ranked_queue.columns
    if col in ["gsc_clicks_mar", "gsc_impressions_mar", "march_outcome"]
]

print("Future/label-derived columns found:", future_columns)

Weak picks in Top-20: 20


,client_hash_id,content_hash_id,gsc_impressions,ctr,action_score,reason_code,confidence_note,what_would_make_it_wrong
0,client_a80fca3f171ed1de,content_6d39e678ecda2135,11.0,0.0,0.0,CTR_OPPORTUNITY,Medium confidence: enough search impressions f...,CTR may be unavailable or too weak to confirm ...
1,client_3ffa76342f366962,content_c2eed5ce37894647,2.0,0.0,0.0,CTR_OPPORTUNITY,"Low confidence: very few impressions, so the r...",Low search volume may make this look like an o...
2,client_3ffa76342f366962,content_36bee0a093d0711d,3.0,0.0,0.0,CTR_OPPORTUNITY,"Low confidence: very few impressions, so the r...",Low search volume may make this look like an o...
3,client_3ffa76342f366962,content_04bed7e232ca42cd,1.0,0.0,0.0,CTR_OPPORTUNITY,"Low confidence: very few impressions, so the r...",Low search volume may make this look like an o...
4,client_3ffa76342f366962,content_0239c0a89e9454c5,1.0,0.0,0.0,CTR_OPPORTUNITY,"Low confidence: very few impressions, so the r...",Low search volume may make this look like an o...
5,client_3ffa76342f366962,content_4e2d90da35481342,3.0,0.0,0.0,CTR_OPPORTUNITY,"Low confidence: very few impressions, so the r...",Low search volume may make this look like an o...
6,client_3ffa76342f366962,content_cfbf60a6d1558926,1.0,0.0,0.0,CTR_OPPORTUNITY,"Low confidence: very few impressions, so the r...",Low search volume may make this look like an o...
7,client_3ffa76342f366962,content_22c76a7e96510f57,1.0,0.0,0.0,CTR_OPPORTUNITY,"Low confidence: very few impressions, so the r...",Low search volume may make this look like an o...
8,client_3ffa76342f366962,content_5736779b45111bab,1.0,0.0,0.0,CTR_OPPORTUNITY,"Low confidence: very few impressions, so the r...",Low search volume may make this look like an o...
9,client_3ffa76342f366962,content_8afd5f61282636ce,2.0,0.0,0.0,CTR_OPPORTUNITY,"Low confidence: very few impressions, so the r...",Low search volume may make this look like an o...


Future/label-derived columns found: []


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.